In [1]:
import pandas as pd

In [2]:
df = pd.read_json("hf://datasets/UniversalCEFR/learn_welsh_cy/learn-welsh-cy.json")
df.head()

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x8b in position 1: invalid start byte

In [15]:
# Pipeline
from transformers import pipeline
pipe = pipeline("text-classification", model="UniversalCEFR/ModernBERT-base-cefr-all-classifier", batch_size=8)

Device set to use cpu


In [16]:
predictions = pipe(df["text"].tolist(), batch_size=8)

In [18]:
df["predicted_level"] = [p["label"] for p in predictions]
df["confidence"] = [p["score"] for p in predictions]

In [19]:
df

,title,lang,source_name,format,category,cefr_level,license,text,predicted_level,confidence
0,Uned 1 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Helô, Eryl dw i. Pwy dych chi?\nB: Bore da,...",A2,0.974658
1,Uned 1 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: O na, yr heddlu! (Stopio'r car)\nB: Hello, ...",A1,0.864848
2,Uned 1 - Sgwrs 3,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Bore da. Sut dych chi?\nB: Iawn, ond wedi b...",A2,0.999449
3,Uned 2 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"Ceri: Noswaith dda, Eryl. Sut wyt ti?\nEryl: D...",A2,0.965478
4,Uned 2 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,A: Bore da.\nB: Hmff.\nA: Sut dych chi heddiw?...,A2,0.613251
...,...,...,...,...,...,...,...,...,...,...
1367,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allech chi gyrraedd yn gynnar?,A2,0.999998
1368,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allet ti gyrraedd yn gynnar?,A2,0.999999
1369,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allai hi gyrraedd yn gynnar?,A2,0.999997
1370,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allen nhw gyrraedd yn gynnar?,A2,0.999954


In [22]:
from sklearn.metrics import classification_report, confusion_matrix
df["predicted_label"] = [p["label"] for p in pipe(df["text"].tolist())]
print(classification_report(df["cefr_level"], df["predicted_label"]))

              precision    recall  f1-score   support

          A1       0.94      0.89      0.92       764
          A2       0.87      0.90      0.89       608
          B1       0.00      0.00      0.00         0
          B2       0.00      0.00      0.00         0

    accuracy                           0.89      1372
   macro avg       0.45      0.45      0.45      1372
weighted avg       0.91      0.89      0.90      1372



c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being 

In [23]:
from sklearn.metrics import f1_score

macro = f1_score(df["cefr_level"], df["predicted_label"], average="macro")
weighted = f1_score(df["cefr_level"], df["predicted_label"], average="weighted")
print(f"Macro-F1: {macro:.3f}, Weighted-F1: {weighted:.3f}")

Macro-F1: 0.450, Weighted-F1: 0.902
